# Apex Legends Tracker - ML Notebook
Tento notebook trenuje dva modely:
- klasifikace ranku
- regrese damage per game

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('data/players.csv')
print('Rows:', len(df))
df.head()

In [ ]:
required_cols = ['level', 'rank_score', 'kills', 'damage', 'headshots', 'games_played', 'wins', 'kdr', 'damage_per_game', 'rank']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns in data/players.csv: {missing}')

df = df.dropna(subset=['rank'])
for col in ['level', 'rank_score', 'kills', 'damage', 'headshots', 'games_played', 'wins', 'kdr', 'damage_per_game']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

df = df[df['games_played'] > 0].copy()
print('Rows after cleaning:', len(df))

In [ ]:
feature_columns = ['level', 'rank_score', 'kills', 'damage', 'headshots', 'games_played', 'wins', 'kdr']
X = df[feature_columns]

label_encoder = LabelEncoder()
y_rank = label_encoder.fit_transform(df['rank'])
y_damage = df['damage_per_game']

In [ ]:
X_train, X_test, y_rank_train, y_rank_test = train_test_split(X, y_rank, test_size=0.2, random_state=42, stratify=y_rank)
_, _, y_damage_train, y_damage_test = train_test_split(X, y_damage, test_size=0.2, random_state=42)

rank_model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
rank_model.fit(X_train, y_rank_train)
rank_pred = rank_model.predict(X_test)
rank_acc = accuracy_score(y_rank_test, rank_pred)

print(f'Rank accuracy: {rank_acc:.4f}')
print(classification_report(y_rank_test, rank_pred, target_names=label_encoder.classes_))

In [ ]:
damage_model = RandomForestRegressor(n_estimators=300, random_state=42)
damage_model.fit(X_train, y_damage_train)
damage_pred = damage_model.predict(X_test)
mae = mean_absolute_error(y_damage_test, damage_pred)

print(f'Damage MAE: {mae:.2f}')

plt.figure(figsize=(8, 6))
plt.scatter(y_damage_test, damage_pred, alpha=0.45)
plt.xlabel('True damage/game')
plt.ylabel('Predicted damage/game')
plt.title('Damage Regression: True vs Predicted')
plt.show()

In [ ]:
bundle = {
    'rank_model': rank_model,
    'damage_model': damage_model,
    'label_encoder': label_encoder,
    'feature_columns': feature_columns,
}

joblib.dump(bundle, 'model/model.pkl')
print('Saved model/model.pkl')